# Myopic policy examples

This notebook shows two current `MyopicPayoffPolicy` workflows:

- one verbose simulator step from a random directed graph with 10 nodes
- many episodes from empty 10-node networks, with social welfare plotted over time


## Imports and helpers


In [ ]:
import sys
from pathlib import Path

import igraph as ig
import matplotlib.pyplot as plt
import numpy as np

try:
    import scienceplots  # noqa: F401
except ModuleNotFoundError:
    scienceplots = None
else:
    plt.style.use("science")

project_root = Path.cwd()
if not (project_root / "network_creation").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from network_creation.core.agent import Agent
from network_creation.core.environment import Environment
from network_creation.core.graph import create_empty_graph
from network_creation.core.model import ActorScheduler
from network_creation.core.observability import ObservationSampling
from network_creation.core.simulator import Simulator
from network_creation.observers.grimoire import OBSERVER_GRIMOIRE
from network_creation.plotting.episode_series import plot_episode_series_observer
from network_creation.plotting.graph_history import (
    build_circle_layout,
    plot_graph_history,
)
from network_creation.policies.myopic_payoff import MyopicPayoffPolicy

ActionPrintObserver = OBSERVER_GRIMOIRE["action_print"]
NormalizedWelfareObserver = OBSERVER_GRIMOIRE["normalized_welfare"]
StateHistoryObserver = OBSERVER_GRIMOIRE["state_history"]

In [ ]:
def make_myopic_agents(
    n_nodes: int,
    visibility_distance: int,
    seed_start: int,
    **policy_kwargs,
) -> tuple[Agent, ...]:
    return tuple(
        Agent(
            id=node_id,
            visibility_distance=visibility_distance,
            policy=MyopicPayoffPolicy(
                seed=seed_start + 1_000 + node_id,
                **policy_kwargs,
            ),
            seed=seed_start + node_id,
        )
        for node_id in range(n_nodes)
    )

In [ ]:
benefit_to_cost_ratio = 0.5

In [ ]:
n_nodes = 10
policy_kwargs = {
    "benefit_to_cost_ratio": benefit_to_cost_ratio,
    "distance_discount": 0.9,
    "bridge_probability": 0.4,
}

In [ ]:
# Normalized welfare no longer needs explicit normalization bounds.

## 1. One verbose step on a random graph

The cell below starts from one random directed graph, runs exactly one simultaneous simulator step with `MyopicPayoffPolicy`, and prints the chosen decisions through `ActionPrintObserver`.


In [ ]:
random_graph = ig.Graph.Erdos_Renyi(n=n_nodes, p=0.1, directed=True, loops=False)

verbose_history_observer = StateHistoryObserver(
    observation_sampling=ObservationSampling.periodic(1),
    keep_runs=False,
)
verbose_simulator = Simulator(
    env=Environment(random_graph, seed=11),
    agents=make_myopic_agents(
        n_nodes=n_nodes,
        visibility_distance=2,
        seed_start=101,
        **policy_kwargs,
    ),
    actor_scheduler=ActorScheduler(n_agents=n_nodes, n_actors=n_nodes, seed=211),
    max_steps=1,
    rng=np.random.default_rng(211),
    observers=(
        ActionPrintObserver(observation_sampling=ObservationSampling.periodic(1), print_empty_steps=True),
        verbose_history_observer,
    ),
)

print("Initial adjacency matrix:")
print(np.asarray(random_graph.get_adjacency().data, dtype=np.int8))

In [ ]:
verbose_simulator.run(n_episodes=1, reset_observers=True)
verbose_history = verbose_history_observer.get()
print("\nFinal adjacency matrix:")
print(np.asarray(verbose_history.graphs[-1].get_adjacency().data, dtype=np.int8))

In [ ]:
layout = build_circle_layout(n_nodes=n_nodes)
figure, _ = plot_graph_history(
    steps=verbose_history.steps,
    graphs=verbose_history.graphs,
    layout=layout,
    figure_title="Random 10-node graph before and after one verbose myopic step",
    max_cols=2,
)
plt.show()

## 2. Welfare from empty networks across many episodes

This section starts every episode from the empty directed graph on the same 10 nodes, runs many episodes under `MyopicPayoffPolicy`, and plots the resulting normalized-welfare trajectories.


In [ ]:
welfare_observer = NormalizedWelfareObserver(
    observation_sampling=ObservationSampling.periodic(1),
    benefit_to_cost_ratio=policy_kwargs["benefit_to_cost_ratio"],
    distance_discount=policy_kwargs["distance_discount"],
    aggregations=("mean", "std", "runs"),
)

welfare_simulator = Simulator(
    env=Environment(create_empty_graph(n_nodes), seed=307),
    agents=make_myopic_agents(
        n_nodes=n_nodes,
        visibility_distance=n_nodes,
        seed_start=401,
        **policy_kwargs,
    ),
    actor_scheduler=ActorScheduler(n_agents=n_nodes, n_actors=n_nodes, seed=503),
    max_steps=360,
    rng=np.random.default_rng(503),
    observers=(welfare_observer,),
)

In [ ]:
welfare_simulator.run(n_episodes=32, reset_observers=True)

In [ ]:
figure, axis = plt.subplots(figsize=(12, 5))
plot_episode_series_observer(
    observer=welfare_observer,
    ax=axis,
    show_runs=True,
    color="#1f4e79",
    band_color="#1f4e79",
    label="mean normalized welfare",
)
axis.set_title(f"Normalized welfare from empty {n_nodes}-node networks")
plt.show()

# 3. Last Graph

In [ ]:
n_nodes = 10
policy_kwargs = {
    "benefit_to_cost_ratio": benefit_to_cost_ratio,
    "distance_discount": 0.9,
    "bridge_probability": 0.4,
}

In [ ]:
verbose_history_observer = StateHistoryObserver(
    observation_sampling=ObservationSampling.terminal(),
    keep_runs=False,
)
verbose_simulator = Simulator(
    env=Environment(create_empty_graph(n_nodes), seed=11),
    agents=make_myopic_agents(
        n_nodes=n_nodes,
        visibility_distance=2,
        seed_start=101,
        **policy_kwargs,
    ),
    actor_scheduler=ActorScheduler(n_agents=n_nodes, n_actors=n_nodes, seed=211),
    max_steps=360*10,
    rng=np.random.default_rng(211),
    observers=(
        verbose_history_observer,
    ),
)

verbose_simulator.run(n_episodes=32, reset_observers=True)
verbose_history = verbose_history_observer.get()

In [ ]:
layout = build_circle_layout(n_nodes=n_nodes)
figure, _ = plot_graph_history(
    steps=verbose_history.steps,
    graphs=verbose_history.graphs,
    layout=layout,
    figure_title="Last graph",
    max_cols=1,
)
plt.show()